In [ ]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only
!git log -1 --oneline


In [ ]:
!pwd
!ls
!nvidia-smi -L


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch

train_device = "0,1" if torch.cuda.device_count() >= 2 else ("0" if torch.cuda.is_available() else "cpu")
print(f"Training device: {train_device} | cuda_count={torch.cuda.device_count()}")

!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models \
  --model yolov8s.pt \
  --img_size 640 \
  --epochs 100 \
  --batch_size -1 \
  --workers 8 \
  --device {train_device} \
  --cache \
  --run_name yolov8_t4x2

In [ ]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --checkpoint ./models/best.pt \
  --results_dir ./results \
  --annotation ./public/annotations/val.json \
  --device 0 \
  --save_vis

In [ ]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

In [ ]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/kaggle/working/O_D')
out_zip = Path('/kaggle/working/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')
